# Time-to-Fill & Funnel Analysis

## Objective
Identify bottlenecks in the recruiting process and understand factors that drive time-to-fill:
- Map candidate flow through each stage
- Calculate conversion rates and time spent at each stage
- Identify where delays occur
- Segment by role, level, department

## Key Questions
1. Where are the major bottlenecks in our recruiting process?
2. Which roles/departments take longest to fill?
3. What factors predict extended time-to-fill?
4. How can we accelerate without sacrificing quality?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load Data

In [ ]:
# Load datasets
requisitions_df = pd.read_csv('../data/requisitions.csv')
candidates_df = pd.read_csv('../data/candidates.csv')
interviews_df = pd.read_csv('../data/interviews.csv')
hires_df = pd.read_csv('../data/hires.csv')

# Convert dates
requisitions_df['opened_date'] = pd.to_datetime(requisitions_df['opened_date'])
requisitions_df['filled_date'] = pd.to_datetime(requisitions_df['filled_date'])
candidates_df['applied_date'] = pd.to_datetime(candidates_df['applied_date'])
interviews_df['interview_date'] = pd.to_datetime(interviews_df['interview_date'])
hires_df['hire_date'] = pd.to_datetime(hires_df['hire_date'])

print(f"📊 Data loaded:")
print(f"   Requisitions: {len(requisitions_df):,}")
print(f"   Candidates: {len(candidates_df):,}")
print(f"   Interviews: {len(interviews_df):,}")
print(f"   Hires: {len(hires_df):,}")

print("\n📋 Requisitions preview:")
requisitions_df.head()

## 2. Overall Time-to-Fill Analysis

In [ ]:
# Analyze filled requisitions only
filled_reqs = requisitions_df[requisitions_df['status'] == 'Filled'].copy()

print("⏱️ Time-to-Fill Summary:\n")
print(f"Total Requisitions: {len(requisitions_df)}")
print(f"Filled: {len(filled_reqs)} ({len(filled_reqs)/len(requisitions_df)*100:.1f}%)")
print(f"Open: {len(requisitions_df[requisitions_df['status'] == 'Open'])}")
print(f"Cancelled: {len(requisitions_df[requisitions_df['status'] == 'Cancelled'])}")

print(f"\nTime-to-Fill Statistics (filled reqs only):")
print(f"   Mean: {filled_reqs['time_to_fill_days'].mean():.1f} days")
print(f"   Median: {filled_reqs['time_to_fill_days'].median():.1f} days")
print(f"   Std Dev: {filled_reqs['time_to_fill_days'].std():.1f} days")
print(f"   Min: {filled_reqs['time_to_fill_days'].min():.0f} days")
print(f"   Max: {filled_reqs['time_to_fill_days'].max():.0f} days")

# Percentiles
print(f"\nPercentiles:")
for p in [25, 50, 75, 90]:
    print(f"   {p}th: {filled_reqs['time_to_fill_days'].quantile(p/100):.1f} days")

In [ ]:
# Distribution of time-to-fill
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
ax1.hist(filled_reqs['time_to_fill_days'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(filled_reqs['time_to_fill_days'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {filled_reqs["time_to_fill_days"].mean():.1f} days')
ax1.axvline(filled_reqs['time_to_fill_days'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {filled_reqs["time_to_fill_days"].median():.1f} days')
ax1.set_xlabel('Time-to-Fill (days)')
ax1.set_ylabel('Number of Requisitions')
ax1.set_title('Distribution of Time-to-Fill')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot
ax2.boxplot(filled_reqs['time_to_fill_days'], vert=True, patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
ax2.set_ylabel('Time-to-Fill (days)')
ax2.set_title('Time-to-Fill Box Plot')
ax2.set_xticklabels(['All Requisitions'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Time-to-Fill by Department

In [ ]:
# Time-to-fill by department
dept_ttf = filled_reqs.groupby('department').agg({
    'time_to_fill_days': ['mean', 'median', 'std', 'count']
}).round(1)

dept_ttf.columns = ['mean_ttf', 'median_ttf', 'std_ttf', 'n_filled']
dept_ttf = dept_ttf.sort_values('mean_ttf', ascending=False)

print("🏢 Time-to-Fill by Department:\n")
print(dept_ttf)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(dept_ttf))
ax.barh(x, dept_ttf['mean_ttf'], alpha=0.7, color='steelblue', label='Mean')
ax.scatter(dept_ttf['median_ttf'], x, color='red', s=100, zorder=5, label='Median', marker='D')

# Add sample size labels
for i, (idx, row) in enumerate(dept_ttf.iterrows()):
    ax.text(row['mean_ttf'] + 1, i, f"n={int(row['n_filled'])}", 
            va='center', fontsize=9)

# Reference line at overall mean
ax.axvline(x=filled_reqs['time_to_fill_days'].mean(), color='gray', 
           linestyle='--', alpha=0.7, label='Overall Mean')

ax.set_yticks(x)
ax.set_yticklabels(dept_ttf.index)
ax.set_xlabel('Time-to-Fill (days)')
ax.set_title('Time-to-Fill by Department', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Time-to-Fill by Role Level

In [ ]:
# Time-to-fill by level
level_ttf = filled_reqs.groupby('level').agg({
    'time_to_fill_days': ['mean', 'median', 'std', 'count']
}).round(1)

level_ttf.columns = ['mean_ttf', 'median_ttf', 'std_ttf', 'n_filled']

# Order by seniority
level_order = ['Entry', 'Mid', 'Senior', 'Lead/Manager']
level_ttf = level_ttf.reindex(level_order)

print("📊 Time-to-Fill by Role Level:\n")
print(level_ttf)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(level_ttf)))
level_ttf['mean_ttf'].plot(kind='bar', ax=ax1, color=colors, edgecolor='black', alpha=0.8)
ax1.axhline(y=filled_reqs['time_to_fill_days'].mean(), color='red', 
            linestyle='--', linewidth=2, label='Overall Mean')
ax1.set_xlabel('Role Level')
ax1.set_ylabel('Average Time-to-Fill (days)')
ax1.set_title('Time-to-Fill by Seniority Level')
ax1.legend()
ax1.set_xticklabels(level_ttf.index, rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(level_ttf['mean_ttf']):
    ax1.text(i, v + 1, f"{v:.1f}", ha='center', fontweight='bold')

# Box plot comparison
level_data = [filled_reqs[filled_reqs['level'] == level]['time_to_fill_days'].values 
              for level in level_order]

bp = ax2.boxplot(level_data, labels=level_order, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_xlabel('Role Level')
ax2.set_ylabel('Time-to-Fill (days)')
ax2.set_title('Time-to-Fill Distribution by Level')
ax2.set_xticklabels(level_order, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Candidate Funnel Analysis

In [ ]:
# Define funnel stages in order
stage_order = [
    'Applied',
    'Screening', 
    'Phone Screen',
    'Technical Interview',
    'Onsite Interview',
    'Offer',
    'Hired'
]

# Count candidates at each stage
stage_counts = candidates_df['current_stage'].value_counts()

# Create funnel data
funnel_data = []
for stage in stage_order:
    if stage in stage_counts:
        funnel_data.append(stage_counts[stage])
    else:
        funnel_data.append(0)

# Calculate conversion rates
total_applied = len(candidates_df)
funnel_df = pd.DataFrame({
    'stage': stage_order,
    'count': funnel_data,
    'pct_of_applied': [c/total_applied*100 for c in funnel_data]
})

# Calculate stage-to-stage conversion
funnel_df['conversion_to_next_%'] = 0.0
for i in range(len(funnel_df) - 1):
    if funnel_df.loc[i, 'count'] > 0:
        funnel_df.loc[i, 'conversion_to_next_%'] = (funnel_df.loc[i+1, 'count'] / funnel_df.loc[i, 'count'] * 100)

funnel_df = funnel_df.round(2)

print("📊 Recruiting Funnel:\n")
print(funnel_df)
print(f"\n💡 Overall conversion rate: {funnel_data[-1]/funnel_data[0]*100:.2f}% (Applied → Hired)")

In [ ]:
# Visualize funnel
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Funnel chart (bar)
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(funnel_df)))
y_pos = range(len(funnel_df))

bars = ax1.barh(y_pos, funnel_df['count'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(funnel_df['stage'])
ax1.set_xlabel('Number of Candidates')
ax1.set_title('Recruiting Funnel: Candidate Flow', fontsize=14, fontweight='bold')
ax1.invert_yaxis()  # Top to bottom flow
ax1.grid(axis='x', alpha=0.3)

# Add count labels
for i, (count, pct) in enumerate(zip(funnel_df['count'], funnel_df['pct_of_applied'])):
    ax1.text(count + 10, i, f"{int(count)} ({pct:.1f}%)", 
            va='center', fontsize=10, fontweight='bold')

# Conversion rates
conversion_data = funnel_df[funnel_df['conversion_to_next_%'] > 0]
ax2.barh(range(len(conversion_data)), conversion_data['conversion_to_next_%'], 
         color='green', alpha=0.7, edgecolor='black')
ax2.set_yticks(range(len(conversion_data)))
ax2.set_yticklabels([f"{s} → Next" for s in conversion_data['stage']])
ax2.set_xlabel('Conversion Rate (%)')
ax2.set_title('Stage-to-Stage Conversion Rates', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, pct in enumerate(conversion_data['conversion_to_next_%']):
    ax2.text(pct + 2, i, f"{pct:.1f}%", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Interview Activity Analysis

In [ ]:
# Interviews per candidate
interviews_per_candidate = interviews_df.groupby('candidate_id').size()

print("📋 Interview Activity:\n")
print(f"Total interviews conducted: {len(interviews_df):,}")
print(f"Candidates who interviewed: {len(interviews_per_candidate):,}")
print(f"Average interviews per candidate: {interviews_per_candidate.mean():.2f}")
print(f"Median interviews per candidate: {interviews_per_candidate.median():.0f}")

# Interview distribution
interview_dist = interviews_per_candidate.value_counts().sort_index()
print(f"\nInterview count distribution:")
for count, freq in interview_dist.items():
    print(f"   {count} interview(s): {freq} candidates ({freq/len(interviews_per_candidate)*100:.1f}%)")

In [ ]:
# Interviews by type
interview_types = interviews_df['interview_type'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Interview types
interview_types.plot(kind='barh', ax=ax1, color='purple', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Number of Interviews')
ax1.set_title('Interviews by Type', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Interviews per candidate histogram
ax2.hist(interviews_per_candidate, bins=range(1, interviews_per_candidate.max()+2), 
         edgecolor='black', alpha=0.7, color='teal')
ax2.axvline(interviews_per_candidate.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {interviews_per_candidate.mean():.2f}')
ax2.set_xlabel('Number of Interviews per Candidate')
ax2.set_ylabel('Number of Candidates')
ax2.set_title('Distribution: Interviews per Candidate', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Time Analysis: Application to Interview to Hire

In [ ]:
# Calculate time from application to first interview
first_interviews = interviews_df.sort_values('interview_date').groupby('candidate_id').first().reset_index()
first_interviews = first_interviews.merge(candidates_df[['candidate_id', 'applied_date']], on='candidate_id')
first_interviews['days_to_first_interview'] = (first_interviews['interview_date'] - first_interviews['applied_date']).dt.days

# Calculate time from application to hire
hired_candidates = candidates_df[candidates_df['current_stage'] == 'Hired'][['candidate_id', 'applied_date']]
hired_candidates = hired_candidates.merge(hires_df[['candidate_id', 'hire_date']], on='candidate_id')
hired_candidates['days_to_hire'] = (hired_candidates['hire_date'] - hired_candidates['applied_date']).dt.days

print("⏱️ Time-based Metrics:\n")
print(f"Application → First Interview:")
print(f"   Mean: {first_interviews['days_to_first_interview'].mean():.1f} days")
print(f"   Median: {first_interviews['days_to_first_interview'].median():.1f} days")

print(f"\nApplication → Hire (for hired candidates):")
print(f"   Mean: {hired_candidates['days_to_hire'].mean():.1f} days")
print(f"   Median: {hired_candidates['days_to_hire'].median():.1f} days")
print(f"   Min: {hired_candidates['days_to_hire'].min():.0f} days")
print(f"   Max: {hired_candidates['days_to_hire'].max():.0f} days")

In [ ]:
# Visualize time distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Days to first interview
ax1.hist(first_interviews['days_to_first_interview'], bins=20, 
         edgecolor='black', alpha=0.7, color='orange')
ax1.axvline(first_interviews['days_to_first_interview'].mean(), 
            color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {first_interviews["days_to_first_interview"].mean():.1f} days')
ax1.set_xlabel('Days from Application to First Interview')
ax1.set_ylabel('Number of Candidates')
ax1.set_title('Time to First Interview', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Days to hire
ax2.hist(hired_candidates['days_to_hire'], bins=20, 
         edgecolor='black', alpha=0.7, color='green')
ax2.axvline(hired_candidates['days_to_hire'].mean(), 
            color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {hired_candidates["days_to_hire"].mean():.1f} days')
ax2.set_xlabel('Days from Application to Hire')
ax2.set_ylabel('Number of Hires')
ax2.set_title('Time-to-Hire (Application → Offer Accept)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Bottleneck Identification

In [ ]:
# Identify slowest departments
slow_depts = dept_ttf.nlargest(3, 'mean_ttf')

print("="*70)
print("🚨 KEY BOTTLENECKS IDENTIFIED")
print("="*70)

print("\n1️⃣ SLOWEST DEPARTMENTS:\n")
for idx, row in slow_depts.iterrows():
    print(f"   {idx}:")
    print(f"      Average: {row['mean_ttf']:.1f} days (vs. overall {filled_reqs['time_to_fill_days'].mean():.1f})")
    print(f"      {row['mean_ttf'] - filled_reqs['time_to_fill_days'].mean():.1f} days SLOWER than average")
    print(f"      Sample size: {int(row['n_filled'])} requisitions")
    print()

# Identify funnel drop-offs
biggest_drops = funnel_df.nsmallest(3, 'conversion_to_next_%')
biggest_drops = biggest_drops[biggest_drops['conversion_to_next_%'] > 0]

print("\n2️⃣ BIGGEST FUNNEL DROP-OFFS:\n")
for idx, row in biggest_drops.iterrows():
    next_stage = funnel_df.loc[idx+1, 'stage'] if idx+1 < len(funnel_df) else 'End'
    print(f"   {row['stage']} → {next_stage}:")
    print(f"      Conversion: {row['conversion_to_next_%']:.1f}%")
    print(f"      Lost: {int(row['count'] - funnel_df.loc[idx+1, 'count'] if idx+1 < len(funnel_df) else 0)} candidates")
    print()

# Time delays
p90_time_to_interview = first_interviews['days_to_first_interview'].quantile(0.9)
slow_to_interview = first_interviews[first_interviews['days_to_first_interview'] > p90_time_to_interview]

print("\n3️⃣ SCHEDULING DELAYS:\n")
print(f"   90th percentile time to first interview: {p90_time_to_interview:.0f} days")
print(f"   Candidates waiting >{p90_time_to_interview:.0f} days: {len(slow_to_interview)} ({len(slow_to_interview)/len(first_interviews)*100:.1f}%)")
print(f"   ⚠️ Risk: Long waits → candidate drop-off")

print("\n" + "="*70)

## 9. Recommendations

In [ ]:
print("="*70)
print("💡 RECOMMENDATIONS")
print("="*70)

print("\n🎯 QUICK WINS (0-30 days):\n")
print("   1. Set SLA: First interview within 7 days of application")
print("   2. Automate interview scheduling (Calendly, GoodTime)")
print("   3. Create standardized interview guides by role")
print(f"   4. Fast-track referrals (currently take {filled_reqs['time_to_fill_days'].mean():.0f} days avg)")

print("\n🏗️ PROCESS IMPROVEMENTS (30-90 days):\n")
print("   1. Reduce interview loops for entry-level roles")
print("   2. Implement panel interviews (fewer rounds, same quality)")
print(f"   3. Address slow departments: {', '.join(slow_depts.head(2).index)}")
print("   4. Train hiring managers on timely decision-making")

print("\n📊 METRICS TO TRACK:\n")
print("   • Days to first interview (target: <7)")
print("   • Days between interview rounds (target: <5)")
print("   • Hiring manager response time (target: <48 hours)")
print("   • Candidate drop-off by stage")
print("   • Time-to-fill by department/level (monthly)")

print("\n🎯 GOALS:\n")
current_avg = filled_reqs['time_to_fill_days'].mean()
print(f"   Current average TTF: {current_avg:.0f} days")
print(f"   Target (realistic): {current_avg * 0.85:.0f} days (-15%)")
print(f"   Target (stretch): {current_avg * 0.70:.0f} days (-30%)")

print("\n" + "="*70)

## Next Steps

1. **Deep Dive**: Analyze slowest departments individually
2. **Interviews**: Survey hiring managers and recruiters on bottlenecks
3. **Pilot**: Test process improvements on 1-2 teams first
4. **Monitor**: Track weekly metrics during improvement initiatives
5. **Iterate**: Adjust based on what works

## Related Analyses
- `01_source_of_hire_analysis.ipynb` - Channel effectiveness
- `04_interview_effectiveness.ipynb` - Interview quality vs. quantity
- `03_quality_of_hire_prediction.ipynb` - Does faster = worse quality?